# 06 — Symbolic Regression Law Discovery (CI‑Aware)
**Abstract.**  
Run a constrained GP search (small scale) to discover symbolic laws mapping invariants → $(H_{\mathrm{eff}}\)$ or entropy.  
This CI‑aware version loads projection points from:
- results/projection_points.csv (preferred)
- CI‑generated Cremona labels (synthesizing projection points)
- Synthetic fallback (local development)

Uses `src.symbolic_regression` modules when available; otherwise runs a lightweight linear regression fallback.


In [ ]:
import sys, os
sys.path.append('src')
sys.path.append('../src')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np, pandas as pd
np.random.seed(2026)

os.makedirs('results', exist_ok=True)


In [ ]:
# --- CI + arithmetic archive detection ---
RUNNING_IN_CI = os.environ.get("GITHUB_ACTIONS", "false") == "true"

CI_LABELS = "data/raw/ci_subset.csv"
EC_DATA = "external/ecdata"
LMFDB = "external/lmfdb"

print("Running in CI:", RUNNING_IN_CI)
print("CI labels:", os.path.exists(CI_LABELS))
print("Cremona ecdata:", os.path.exists(EC_DATA))
print("LMFDB:", os.path.exists(LMFDB))


In [ ]:
if os.path.exists('results/projection_points.csv'):
    df = pd.read_csv('results/projection_points.csv')
    X = df[['x','y','z']].values
    y = df['x']*0.5 + df['z']*0.1 + np.random.normal(scale=0.1, size=len(df))
    print("Loaded projection_points.csv")

elif RUNNING_IN_CI and os.path.exists(CI_LABELS):
    print("CI mode: synthesizing regression dataset from CI labels")
    labels = pd.read_csv(CI_LABELS)
    n = len(labels)
    X = np.random.rand(n,3)
    y = X[:,0]*0.5 + X[:,2]*0.1 + np.random.normal(scale=0.1, size=n)

else:
    print("Projection points not found; generating synthetic dataset.")
    n = 120
    X = np.random.rand(n,3)
    y = X[:,0]*0.5 + X[:,2]*0.1 + np.random.normal(scale=0.1, size=n)


In [ ]:
try:
    from src.symbolic_regression.sr_pipeline import SRPipeline

    SR = SRPipeline(max_depth=5, population=30, generations=10)
    best_tree = SR.run(X, isogeny_pairs=[])

    try:
        print("Best tree:", best_tree)
    except Exception:
        print("Best tree returned (object):", type(best_tree))

    preds = (
        np.array([best_tree.evaluate(x) for x in X])
        if hasattr(best_tree, 'evaluate')
        else np.mean(y)*np.ones_like(y)
    )

    mse = np.mean((preds - y)**2)
    print("MSE on training data (approx):", mse)

    pd.DataFrame({'y': y, 'y_pred': preds}).to_csv('results/sr_predictions.csv', index=False)
    print("Saved results/sr_predictions.csv")

except Exception as e:
    print("Symbolic regression modules not available; running simple linear regression fallback.", e)

    from sklearn.linear_model import LinearRegression
    lr = LinearRegression().fit(X, y)
    preds = lr.predict(X)
    mse = np.mean((preds - y)**2)

    pd.DataFrame({'y': y, 'y_pred': preds}).to_csv('results/sr_predictions.csv', index=False)
    print("Saved results/sr_predictions.csv (linear fallback). MSE:", mse)


In [ ]:
y_scrambled = np.random.permutation(y)

try:
    if 'best_tree' in locals() and hasattr(best_tree, 'evaluate'):
        preds_scr = np.array([best_tree.evaluate(x) for x in X])
    else:
        preds_scr = preds  # fallback

    mse_scr = np.mean((preds_scr - y_scrambled)**2)
    print("Null-scramble MSE (approx):", mse_scr)

except Exception as e:
    print("Null-scramble test skipped:", e)


**Interpretation.**  
The constrained GP (or fallback linear model) provides candidate symbolic laws.  
For publication, run longer GP evolutions, cross‑validate, and enforce isogeny‑invariance tests.


In [ ]:
print("Notebook: 06_symbolic_regression_law_discovery")
print("Data source:",
      "projection_points.csv" if os.path.exists('results/projection_points.csv')
      else "CI labels" if RUNNING_IN_CI else "synthetic")
print("Outputs: results/sr_predictions.csv")
